# Train VIMO's voice on a free Colab GPU

**What this does, in plain words:** it takes the sentences you recorded in the VIMO app (Settings → Train VIMO), teaches a small speech model (Whisper tiny) your voices, your cow names and dairy words, checks how often it gets words wrong (word error rate, WER), and — only if the new model is better than the old one — publishes it as a language pack. Your phone then downloads it and VIMO listens offline.

This is the same training as the GitHub workflow, but a GPU makes it much faster.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run each cell from top to bottom (tap ▶).

## 1. Get VIMO's training scripts
This copies the scripts from the VIMO repository into Colab.

In [ ]:
!git clone --depth 1 https://github.com/skwatsonoff/vimo-ranch-management.git
%cd vimo-ranch-management/tools/voice_training
!pip install --quiet -r requirements.txt
!(type gh >/dev/null 2>&1) || (apt-get -qq update && apt-get -qq install -y gh)

## 2. Sign in
Colab asks you to choose your Google account: choose the one that owns the Firebase project **my-ranch-sync** (it reads your recordings from Firestore). Publishing the new pack to GitHub needs a GitHub token saved in Colab's 🔑 Secrets as `GH_TOKEN`; without it, Colab trains and checks only, and the GitHub workflow publishes on its next run.

In [ ]:
from google.colab import auth, userdata
auth.authenticate_user()
import os
os.environ['GOOGLE_CLOUD_PROJECT'] = 'my-ranch-sync'
os.environ['GITHUB_REPOSITORY'] = 'skwatsonoff/vimo-ranch-management'
try:
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
except Exception:
    print('No GH_TOKEN secret: training and checking only.')

## 3. Which training request?
Leave `JOB_ID` empty to take the oldest request you made with **Start training** in the app.

In [ ]:
JOB_ID = ''  # or paste the id shown in the app
import subprocess
if not JOB_ID:
    JOB_ID = subprocess.run(['python', 'train_job.py', 'check'], capture_output=True, text=True).stdout.strip()
print('Job:', JOB_ID or 'none requested — tap Start training in the app first')

## 4. Train, check and publish
A GPU can train longer in the same time, so we allow 120 minutes here. The phone shows each step live: running → evaluating → packaging → ready. If the new model is not better, the old one stays.

In [ ]:
os.environ['VIMO_MAX_MINUTES'] = '120'
os.environ['VIMO_LOG_URL'] = 'Google Colab'
!python train_job.py run "$JOB_ID" 

## 5. The result
`wer` is the share of words the model got wrong on sentences it did not train on (lower is better).

In [ ]:
import json
print(json.dumps(json.load(open('work/train_report.json')), indent=1, ensure_ascii=False))